In [0]:
%sql
USE CATALOG workspace;
USE SCHEMA default;


In [0]:
%sql
CREATE TABLE countries
(country STRING, code STRING, dial_code STRING);

INSERT INTO countries
VALUES 
  ('France', 'FR', '+33'),
  ('Germany', 'DE', '+49'),
  ('United States', 'US', '+1'),
  ('United Kingdom', 'GB', '+44'),
  ('Switzerland', 'CH', '+41');

Stored View (Lebensdauer bis drop view)

In [0]:
CREATE OR REPLACE VIEW germany_switzerland_stored_view AS
SELECT
  *
FROM
  countries
WHERE
  country IN ('Germany', 'Switzerland');

Temporary View (Lebensdauer nur aktive Session)

In [0]:
CREATE OR REPLACE TEMP VIEW france_temporary_view AS
SELECT
  *
FROM
  countries
WHERE
  country IN ('France');

Global Temp View (Lebensdauer bis Cluster Stopp) --> Nicht supportet in Serverless Compute

In [0]:
CREATE OR REPLACE GLOBAL TEMP VIEW germany_global_temp_view AS
SELECT
  *
FROM
  countries
WHERE
  country IN ('Germany');

Views lesen

In [0]:
SELECT * FROM germany_switzerland_stored_view

In [0]:
SELECT * FROM france_temporary_view

In [0]:
-- CLUSTER NEU STARTEN

In [0]:
SELECT * FROM germany_switzerland_stored_view

In [0]:
SELECT * FROM france_temporary_view

dynamic view

In [0]:
%python
# AUSGANGSLAGE (wir können dynamische Views über Metadaten wie z.B: Usergruppen steuern)

# Databricks notebook source
from pyspark.sql.functions import lit

df_groups = spark.sql("""SHOW GROUPS""")

df_users = spark.sql("""SHOW USERS""")

result = spark.createDataFrame([], "name: string, directGroup: boolean, user: string")
for user in df_users.collect():
    df_user_groups = spark.sql(f"""SHOW GROUPS WITH USER `{user.name}`""").withColumn("user", lit(user.name))
    result = result.unionAll(df_user_groups)


display(result)

In [0]:
-- Ausgangslage bilder die Tabelle countries
SELECT
  *
FROM
  countries

Column Level permissions

In [0]:
CREATE OR REPLACE VIEW column_level_stored_view AS
SELECT
  country,
  code,
  CASE WHEN
    is_member('admins') THEN dial_code --admins durch andere gruppe ersetzen
    ELSE '*******'
  END AS dial_code
FROM
  countries

In [0]:
SELECT * FROM column_level_stored_view

Row-level permissions

In [0]:
CREATE OR REPLACE VIEW row_level_stored_view AS
SELECT
  country,
  code,
  dial_code
FROM
  countries
WHERE
  CASE
    WHEN is_member('admins') THEN TRUE   --admins durch andere gruppe ersetzen
    ELSE FALSE
  END;

In [0]:
SELECT * FROM row_level_stored_view

Data masking

In [0]:
CREATE OR REPLACE VIEW data_masked_stored_view AS
SELECT
  country,
  code,
  CASE
    -- 1. Berechtigungsprüfung
    WHEN is_member('admins') THEN dial_code 
    -- 2. Maskierung für alle anderen: Ersetze alle Zahlen nach dem Plus durch ein Sternchen
    ELSE regexp_replace(dial_code, '(?<=\\+)\\d+', '**')
  END as dial_code
FROM
  countries;

In [0]:
SELECT * FROM data_masked_stored_view;

Cleanup

In [0]:
-- aufräumen
DROP VIEW germany_switzerland_stored_view;
DROP VIEW column_level_stored_view;
DROP VIEW row_level_stored_view;
DROP VIEW data_masked_stored_view;
DROP TABLE countries;